# EVALUATION

In [2]:
pip install elasticsearch==8.12.0



  Attempting uninstall: elastic-transport

    Found existing installation: elastic-transport 9.2.1

    Uninstalling elastic-transport-9.2.1:

      Successfully uninstalled elastic-transport-9.2.1

   ---------------------------------------- 0/2 [elastic-transport]
   ---------------------------------------- 0/2 [elastic-transport]
   ---------------------------------------- 0/2 [elastic-transport]
   ---------------------------------------- 0/2 [elastic-transport]
   ---------------------------------------- 0/2 [elastic-transport]
  Attempting uninstall: elasticsearch
   ---------------------------------------- 0/2 [elastic-transport]
    Found existing installation: elasticsearch 9.3.0
   ---------------------------------------- 0/2 [elastic-transport]
    Uninstalling elasticsearch-9.3.0:
   ---------------------------------------- 0/2 [elastic-transport]
      Successfully uninstalled elasticsearch-9.3.0
   ---------------------------------------- 0/2 [elastic-transport]
   ----


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
# get documents
from elasticsearch import Elasticsearch
es = Elasticsearch("http://localhost:9200")

docs = es.search(index="documents", query={"match_all": {}}, size=1000)

In [25]:
hits = docs["hits"]["hits"]
documents = [h["_source"] for h in hits]

len(documents), documents[0].keys()


(10, dict_keys(['title', 'internalName', 'content', 'nlp', 'timestamp']))

In [26]:
# document extraction quality

contents = [d["content"] for d in documents]
total = len(documents)
non_empty = sum(len(c.strip()) > 0 for c in contents)

print("Total documents:", total)
print("Extraction success rate:", non_empty / total)
print("Average content length:", sum(len(c) for c in contents) / total)


Total documents: 10
Extraction success rate: 1.0
Average content length: 14216.7


In [27]:
# NLP enrichment quality

summaries = [d.get("nlp", {}).get("summary") for d in documents]

coverage = sum(s is not None for s in summaries) / len(documents)
print("Summary coverage:", coverage)


Summary coverage: 1.0


In [29]:
for d in documents[:1]:
    print("TITLE:", d["title"])
    print("SUMMARY:", d["nlp"]["summary"])
    print("CONTENT:", d["content"][:400])
    print("="*100)


TITLE: INFO 380 Fall 25 October 15.pdf
SUMMARY:  Workshop Today (Sit with your teams) (Only grab your nametag – grabbing others’ nametags gets you a zero) INFO 380 Information Systems Analysis and Design Vision & Scope BacklogDeliver Solution Iterations Diverge Converge Explore Problems & opportunities Identify themes & needs Diverge Converge Describing the System Vision System/Product Vision Statements... Summarizes the long-term purpose and intent of the system or product. Should reflect a balanced view that will satisfy the expectations of diverse stakeholders. Should be grounded in the realities of existing or anticipated markets, enterprise architectures, the organization's strategic direction, and resource limitations.
CONTENT:  Workshop Today (Sit with your teams) (Only grab your nametag – grabbing others’ nametags gets you a zero) INFO 380 Information Systems Analysis and Design Vision & Scope BacklogDeliver Solution Iterations Diverge Converge Explore Problems & opportunities 

Although every document has had a summary coverage of 1.0, qualitative inspection on the NLP summaries shows that our summaries aren't really summaries. Looking at the NLP worker script, we used TextBlob to extract the first three sentences of the document, which for lecture slides for example only corresponds to title and first bullet points. Therefore our summaries end up nearly identical to the document and don't provide meaningful abstraction.

In [14]:
# Manual labels on what documents are relevant
reference = {
    "machine learning": ["machineLearning.pdf"],
    "INFO 380": [
        "Fall25_INFO380_November24.pdf",
        "INFO380_Fall25_October27.pdf",
        "INFO380_Fall25_October13.pdf",
        "INFO380_Fall25_October29.pdf",
        "INFO380_Fall25_Week1Class1.pdf"
    ],
    "prioritization": ["INFO380_Fall25_October29.pdf"]
}

# helper function for searching documents and comparing vs our manual labels.
def search(query):
    result = es.search(
        index="documents",
        query={"match": {"content": query}},
        size=10
    )
    return [hit["_source"]["title"] for hit in result["hits"]["hits"]]


In [15]:

def compute_precision(retrieved, relevant, k=5):
    return len(set(retrieved[:k]) & set(relevant)) / k

# compute recall
def compute_recall(retrieved, relevant, k=5):
    retrieved_k = set(retrieved[:k])
    relevant_set = set(relevant)
    return len(retrieved_k & relevant_set) / len(relevant_set)

for query, relevant in reference.items():
    retrieved = search(query)
    print(retrieved)
    print()
    print(
        query,
        "Recall@5:", compute_recall(retrieved, relevant, k=5)
    )




['machineLearning.pdf', 'INFO380_Fall25_October13.pdf', 'Fall25_INFO380_November24.pdf']

machine learning Recall@5: 1.0
['INFO380_Fall25_Week1Class1.pdf', 'INFO380_Fall25_October13.pdf', 'Fall25_INFO380_November24.pdf', 'INFO380_Fall25_October 27.pdf', 'machineLearning.pdf']

INFO 380 Recall@5: 0.6
['INFO380_Fall25_October29.pdf', 'INFO380_Fall25_October 27.pdf', 'INFO380_Fall25_October13.pdf', 'INFO380_Fall25_Week1Class1.pdf']

prioritization Recall@5: 1.0


In [18]:
es.delete_by_query(
    index="documents",
    body={
        "query": {
            "match": {
                "title": "machineLearning.pdf"
            }
        }
    }
)


ObjectApiResponse({'took': 140, 'timed_out': False, 'total': 1, 'deleted': 1, 'batches': 1, 'version_conflicts': 0, 'noops': 0, 'retries': {'bulk': 0, 'search': 0}, 'throttled_millis': 0, 'requests_per_second': -1.0, 'throttled_until_millis': 0, 'failures': []})